In [13]:
! pip install langchain_community tiktoken langchain-openai langchain_chroma langchainhub langchain chromadb
! pip install langchain-community pypdf
! pip install fonttools
! pip install python-dotenv
! pip install langchain_huggingface

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

DOC_DIR = "./HandsOnML.pdf"
DB_DIR = "./chroma_db_cosine"

In [16]:
#Load

loader = PyPDFLoader(DOC_DIR)
docs = loader.load()
print(len(docs))
# print(docs[0].page_content[:500])
# print(docs[0].metadata)
print(list(docs[0].metadata.keys()))


564
['producer', 'creator', 'creationdate', 'author', 'moddate', 'title', 'trapped', 'source', 'total_pages', 'page', 'page_label']


In [17]:
#split
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap= 200)
splits = text_splitter.split_documents(docs)
sanitized_splits =[]

for doc in splits:
    clean_text = (
        doc.page_content
            .replace("\x00", "")
            .encode("utf-8",errors= "ignore")
            .decode("utf-8")
            .strip()
    )
    if clean_text:
        sanitized_splits.append(Document(page_content= clean_text, metadata=doc.metadata))
        

print(f"Original splits: {len(splits)} | Sanitized splits: {len(sanitized_splits)}")

# clean_splits = [
#     doc for doc in splits 
#     if isinstance(doc.page_content, str) and doc.page_content.strip()
# ]
# valid_splits = [
#     Document(page_content=str(doc.page_content), metadata=doc.metadata)
#     for doc in clean_splits
#     if doc.page_content is not None and str(doc.page_content).strip() != ""
# ]

# print(len(splits), len(clean_splits), len(valid_splits))

# tokenizer = embeddings._client.tokenizer

# for i, doc in enumerate(valid_splits):
#     try:
#         tokenizer([doc.page_content])
#     except Exception as e:
#         print(f"Failed at index {i}: {e}")
#         print(f"Repr of text: {repr(doc.page_content[:200])}")
#         break
# print("\n",repr(valid_splits[157].page_content))


Original splits: 1455 | Sanitized splits: 1455


In [ ]:
#test
# print(f"{len(docs)} pages -> {len(splits)} chunks")

# lengths = [len(s.page_content) for s in splits]
# print("min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths) // len(lengths))
# assert max(lengths) <= 1000, "a chunk exceeded chunk_size"

# print(splits[0].metadata)
# print(splits[500].metadata)

In [18]:
#embed
import os 
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# 1. Wipe existing DB_DIR so Chroma doesn't append to the old L2 collection
if os.path.exists(DB_DIR):
    shutil.rmtree(DB_DIR)
    print(f"Removed old directory '{DB_DIR}'.")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 32, "normalize_embeddings": True},
)

vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=DB_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)
print("Collection metadata:", vectorstore._collection.metadata)

BATCH_SIZE = 100
for i in range(0, len(sanitized_splits), BATCH_SIZE):
    batch = sanitized_splits[i : i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"Embedded {min(i + BATCH_SIZE, len(sanitized_splits))} / {len(sanitized_splits)} chunks")

print(f"Done! Saved to '{DB_DIR}'.")

Collection metadata: {'hnsw:space': 'cosine'}
Embedded 100 / 1455 chunks
Embedded 200 / 1455 chunks
Embedded 300 / 1455 chunks
Embedded 400 / 1455 chunks
Embedded 500 / 1455 chunks
Embedded 600 / 1455 chunks
Embedded 700 / 1455 chunks
Embedded 800 / 1455 chunks
Embedded 900 / 1455 chunks
Embedded 1000 / 1455 chunks
Embedded 1100 / 1455 chunks
Embedded 1200 / 1455 chunks
Embedded 1300 / 1455 chunks
Embedded 1400 / 1455 chunks
Embedded 1455 / 1455 chunks
Done! Saved to './chroma_db_cosine'.


In [19]:
total_docs = vectorstore._collection.count()
print(f"Total vectors stored in Chroma: {total_docs}")

query = "how do we train a model"
results = vectorstore.similarity_search_with_relevance_scores(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    raw_page = doc.metadata.get("page")
    page_num = raw_page + 1 if isinstance(raw_page, int) else "N/A"
    
    print(f"=== Result {i} | Page: {page_num} | Score: {score:.4f} ===")
    print(doc.page_content)
    print()

Total vectors stored in Chroma: 1455
=== Result 1 | Page: 127 | Score: 0.5131 ===
CHAPTER 4
Training Models
So far we have treated Machine Learning models and their training algorithms mostly
like black boxes. If you went through some of the exercises in the previous chapters,
you may have been surprised by how much you can get done without knowing any‐
thing about what’s under the hood: you optimized a regression system, you improved
a digit image classifier, and you even built a spam classifier from scratch—all this
without knowing how they actually work. Indeed, in many situations you don’t really
need to know the implementation details.
However, having a good understanding of how things work can help you quickly
home in on the appropriate model, the right training algorithm to use, and a good set
of hyperparameters for your task. Understanding what’s under the hood will also help
you debug issues and perform error analysis more efficiently. Lastly, most of the top‐
ics discussed in